<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/torch_ntuple_net_lambda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
if IN_COLAB:
    import os

    if not os.path.exists(os.path.expanduser("~/.ssh/id_rsa")):
        !ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
    else:
        print("Key already exists!")
    !ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts 2>/dev/null
    !cat ~/.ssh/id_rsa.pub

Add key here:
https://github.com/settings/keys

In [ ]:
if IN_COLAB:
    !ssh -T git@github.com
    !git config --global user.email "8896197+MarkusThill@users.noreply.github.com"
    !git config --global user.name "Markus Thill"
    import os

    if not os.path.exists("techdays26"):
        !git clone git@github.com:MarkusThill/techdays26.git
    else:
        !git -C techdays26 pull
    !pip install -e techdays26[dev,lab1]

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

In [ ]:
import copy
import time

import torch
import torch.nn.functional as F

from techdays26 import bitbully_arena as ba
from techdays26.bitbully_arena import format_aggregate_table, get_table_legend
from techdays26.ntuple_network import NTupleNetwork
from techdays26.ntuples import NTUPLE_BITIDX_LIST_150
from techdays26.td_agent import TDConnect4AgentTorch
from techdays26.torch_board import BoardBatch
from techdays26.training import best_afterstate_values

In [ ]:
from techdays26.ntuples import format_ntuple, ntuple_summary

info = ntuple_summary(NTUPLE_BITIDX_LIST_150)
print(
    f"N-tuples: {info['count']} x {info['length']}-tuples  (LUT size: {info.get('lut_size')})"
)
print(f"Hash: {info['hash'][:16]}...")
print()

for i, tup in enumerate(NTUPLE_BITIDX_LIST_150):
    print(f"--- Tuple {i:2d}: {tup} ---")
    print(format_ntuple(tup))
    print()

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent

bitbully_agent_max_depth = BitBully(opening_book="12-ply-dist", tie_break="random")

# Some weaker agents with limited depth
bitbully_agents = {}
for search_depth in (1, 2, 4, 8):
    agent = BitBully(opening_book=None, tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}ply"] = agent

for search_depth in (8, 10):
    agent = BitBully(opening_book="8-ply", tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}-ply-book8ply"] = agent

bitbully_agents[f"bitbully-16ply-book12ply"] = BitBully(
    opening_book="12-ply-dist", tie_break="random", max_depth=16
)
bitbully_agents[f"bitbully-full-strength"] = bitbully_agent_max_depth

In [ ]:
import logging
from techdays26 import bitbully_arena as ba

from techdays26.bitbully_arena import (
    get_table_legend,
    format_aggregate_table,
)


def evaluate(td_agent, rnd_agent, bitbully_agents):
    # Logger is optional; arena is silent by default unless you configure logging.
    logger = logging.getLogger("bitbully.arena")
    logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout

    agent_specs = [
        ba.AgentSpec(
            agent_id=k,
            agent=v,
            colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
            epsilons=(0.00, 0.1, 0.2, 0.3) if "full-strength" in k else (0.00,),
        )
        for k, v in bitbully_agents.items()
    ]
    matchups = (
        [ba.Matchup(yellow_id=k, red_id="ntuple") for k in bitbully_agents.keys()]
        + [ba.Matchup(yellow_id="ntuple", red_id=k) for k in bitbully_agents.keys()]
        + [
            ba.Matchup(yellow_id="ntuple", red_id="random"),
            ba.Matchup(yellow_id="random", red_id="ntuple"),
        ]
    )

    cfg = ba.ArenaConfig(
        agents=(
            *agent_specs,
            ba.AgentSpec(
                agent_id="random",
                agent=rnd_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
                epsilons=(0.00,),
            ),
            ba.AgentSpec(
                agent_id="ntuple",
                agent=td_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both,
                epsilons=(0.00,),
            ),
        ),
        n_games=50,  # n games per pairing per ε per color-swap
        time_control=ba.TimeControl(
            per_move_timeout_s=4.0,  # best-effort (measured after return)
            per_game_budget_s=45.0,  # seconds of total think time
        ),
        matchups=matchups,
        seed=None,
        log_scores=False,  # set True to also call score_all_moves() for logging
        use_tqdm=True,  # optional progress bar
        logger=logger,
    )

    # -----------------------------
    # Run arena
    # -----------------------------

    arena = ba.BitBullyArena()
    result = arena.run(cfg)
    return result

In [ ]:
from techdays26 import utils

commit_sha = utils.get_commit_hash("/content/techdays26" if IN_COLAB else ".")
reqs = utils.get_requirements_string()

In [ ]:
import json
import platform
import sys
from datetime import datetime
from pathlib import Path

from techdays26 import __version__ as techdays26_version
from techdays26.ntuples import ntuple_summary

# ── Device & model ─────────────────────────────────────────────────────────
device = "cuda" if IN_COLAB else "cpu"
pre_trained_model = None  # path to a .pt checkpoint, or None to start fresh

# ── Experiment output ──────────────────────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H-%M")
train_folder = (
    Path("/content/drive/MyDrive/models/" if IN_COLAB else "./") / f"exp_{timestamp}"
)

# ── Training ───────────────────────────────────────────────────────────────
n_steps = 40_000  # environment steps per repeat
n_evaluate = 1_000  # run arena evaluation every n_evaluate steps
n_repeats = 10  # independent runs with the same hyperparameters
B = 20_000  # parallel games (batch size)
epsilon = 0.1  # ε-greedy exploration rate

# ── Learning-rate schedule (exponential decay) ─────────────────────────────
lr_initial = 3e-4
lr_final = 1e-6
gamma = 0.99999  # per-step decay factor  (0.9999 for faster decay in 100k games)

# ── TD(λ) — truncated λ-return ─────────────────────────────────────────────
lam = 0.8  # trace decay  (0 = TD(0),  1 = Monte Carlo)
n_truncate = 5  # lookahead horizon

# ── Target network (Polyak / EMA) ──────────────────────────────────────────
use_target_net = True  # False → bootstrap from online net (simpler, less stable)
use_online_net_for_action = True  # only relevant when use_target_net=True
tau = 0.005  # EMA coefficient (smaller = slower target update)

# ── Misc ────────────────────────────────────────────────────────────────────
use_torch_compile = True  # set False to disable torch.compile (e.g. for debugging)

In [ ]:
# ── Create experiment folder & persist hyperparameters ────────────────────
# (reproducibility bookkeeping — not essential for understanding the algorithm)

train_folder.mkdir(parents=False, exist_ok=False)
ntuple_info = ntuple_summary(NTUPLE_BITIDX_LIST_150)

(train_folder / "0_ntuples.json").write_text(
    json.dumps({"ntuples": NTUPLE_BITIDX_LIST_150, "summary": ntuple_info}, indent=2)
)
(train_folder / "0_params.json").write_text(
    json.dumps(
        {
            "start_time": timestamp,
            "device": device,
            "n_steps": n_steps,
            "n_evaluate": n_evaluate,
            "n_repeats": n_repeats,
            "batch_size": B,
            "epsilon": epsilon,
            "lr_initial": lr_initial,
            "lr_final": lr_final,
            "gamma": gamma,
            "lam": lam,
            "n_truncate": n_truncate,
            "use_target_net": use_target_net,
            "use_online_net_for_action": use_online_net_for_action
            if use_target_net
            else None,
            "tau": tau if use_target_net else None,
            "use_torch_compile": use_torch_compile,
            "ntuple_hash": ntuple_info["hash"],
            "commit_sha": commit_sha,
            "techdays26_version": techdays26_version,
            "python_version": sys.version,
            "pytorch_version": torch.__version__,
        },
        indent=2,
    )
)
root_log = train_folder / "0_log.txt"
root_log.write_text(
    f"techdays26 {techdays26_version} | commit {commit_sha}\n"
    f"host: {platform.node()} | python {sys.version} | torch {torch.__version__}\n"
    f"device: {device} | B={B} | n_steps={n_steps} | lr={lr_initial}→{lr_final}\n"
    f"TD(λ={lam}, n={n_truncate}) | target_net={use_target_net} | tau={tau}\n"
    + get_table_legend()
    + "\n"
)
print(f"Experiment folder: {train_folder}")
print(f"n_repeats={n_repeats}, n_steps={n_steps}, B={B}, device={device}")

In [ ]:
class _TrainingLogger:
    """Handles periodic console logging and arena evaluation.

    Collapsible — not essential for understanding the training algorithm.
    """

    def __init__(
        self, repeat_dir, n_evaluate, n_truncate, n_repeats, repeat_idx, agents
    ):
        self._dir = repeat_dir
        self._n_eval = n_evaluate
        self._log_every = n_evaluate // 10
        self._n_trunc = n_truncate
        self._n_repeats = n_repeats
        self._ri = repeat_idx
        self._agents = agents

        self._metrics_path = repeat_dir / "0_metrics.json"
        self._arena_path = repeat_dir / "0_arena_metrics.json"
        self._log_path = repeat_dir / "0_log.txt"
        self._all_metrics = []
        self._all_arena = []
        self._t0 = time.time()
        self._last_eval_t = time.time()

        # weight-delta state (captured just before opt.step)
        self._W_snap = None
        self._W_mask = None
        self._W_norm = 0.0

    def snapshot_weights(self, net, step):
        """Call just before opt.step() to enable ||ΔW||/||W|| logging."""
        if step % self._log_every == 0:
            with torch.no_grad():
                mask = net.W != 0.0
                snap = net.W[mask].data.clone()
                self._W_snap, self._W_mask, self._W_norm = (
                    snap,
                    mask,
                    snap.norm().item(),
                )

    def __call__(
        self, step, net, opt, loss, done, randomize, update_mask, V_pred, board
    ):
        pfx = f"[R{self._ri}] " if self._n_repeats > 1 else ""

        # ── Console + JSON metrics every log_every steps ──────────────────
        if step % self._log_every == 0:
            now, elapsed = time.time(), time.time() - self._t0
            lr = opt.param_groups[0]["lr"]

            rel_update = dW_norm = 0.0
            if self._W_snap is not None:
                with torch.no_grad():
                    dW = net.W[self._W_mask].data - self._W_snap
                    dW_norm = dW.norm().item()
                    rel_update = (
                        dW_norm / self._W_norm if self._W_norm > 0 else float("inf")
                    )

            grad = net.W.grad
            grad_nz = grad[grad != 0.0] if grad is not None else torch.zeros(0)
            mv_left = board.moves_left

            print(
                f"{pfx}step {step:6d} | {self._fmt(elapsed)} | lr={lr:.3e} | "
                f"loss={loss.item():.5f} | ||ΔW||/||W||={rel_update:.3e} | "
                f"V={V_pred.mean().item():.3f}±{V_pred.std().item():.3f} | "
                f"done={done.float().mean():.2f} | "
                f"moves_left={mv_left.float().mean().item():.1f}"
            )

            m = {
                "step": step,
                "training_elapsed_s": elapsed,
                "training_elapsed": self._fmt(elapsed),
                "wall_time_s": now - self._last_eval_t,
                "lr": lr,
                "loss": loss.item(),
                "rel_weight_update": rel_update,
                "delta_W_norm": dW_norm,
                "W_norm": self._W_norm,
                "V_old_min": V_pred.min().item(),
                "V_old_max": V_pred.max().item(),
                "V_old_mean": V_pred.mean().item(),
                "V_old_std": V_pred.std().item(),
                "V_old_abs_mean": V_pred.abs().mean().item(),
                "grad_nnz": int(grad_nz.shape[0]),
                "grad_mean": grad_nz.mean().item() if grad_nz.numel() > 0 else 0.0,
                "grad_std": grad_nz.std().item() if grad_nz.numel() > 1 else 0.0,
                "update_frac": float(update_mask.float().mean()),
                "done_frac": float(done.float().mean()),
                "randomize_frac": float(randomize.float().mean()),
                "n_wins": int(board.has_win().sum().item()),
                "moves_left_mean": float(mv_left.float().mean()),
                "moves_left_std": float(mv_left.float().std()),
            }
            self._all_metrics.append(m)
            self._safe_write(self._metrics_path, self._all_metrics)

            with self._log_path.open("a") as f:
                sep = "=" * 80
                f.write(
                    f"{sep}\nStep {step:6d} | {m['training_elapsed']} | "
                    f"lr={m['lr']:.3e} | loss={m['loss']:.5f}\n"
                    f"  V: {m['V_old_mean']:.3f}±{m['V_old_std']:.3f}  "
                    f"||ΔW||/||W||={m['rel_weight_update']:.3e}\n"
                    f"  grad_nnz={m['grad_nnz']}  done={m['done_frac']:.2f}  "
                    f"rand={m['randomize_frac']:.2f}\n{sep}\n\n"
                )

        # ── Arena evaluation every n_evaluate steps ───────────────────────
        if step % self._n_eval == 0:
            print(f"{pfx}evaluate at step {step}...")
            weights_path = str(self._dir / f"step_{step}_model_weights.pt")
            net.save(weights_path)
            td_agent = TDConnect4AgentTorch(model_path=weights_path)
            result = evaluate(td_agent, ba.RandomAgent(), self._agents)
            result.save_json(str(self._dir / f"step_{step}_arena_result.json"))

            self._all_arena.append({
                "step": step,
                "training_elapsed_s": time.time() - self._t0,
                "aggregates": self._result_rows(result),
            })
            self._safe_write(self._arena_path, self._all_arena)

            with self._log_path.open("a") as f:
                f.write(
                    "=" * 90 + "\n" + format_aggregate_table(result) + "=" * 90 + "\n\n"
                )

            self._last_eval_t = time.time()

    @staticmethod
    def _fmt(s: float) -> str:
        h, r = divmod(int(s), 3600)
        m, sec = divmod(r, 60)
        return f"{h:02d}:{m:02d}:{sec:02d}"

    @staticmethod
    def _result_rows(result) -> list[dict]:
        rows = []
        for r in result.aggregates:
            score, games = int(r.yellow_wins) - int(r.red_wins), int(r.games)
            rows.append({
                "agent_yellow": r.agent_yellow,
                "agent_red": r.agent_red,
                "epsilon_yellow": float(r.epsilon_yellow),
                "epsilon_red": float(r.epsilon_red),
                "games": games,
                "yellow_wins": int(r.yellow_wins),
                "red_wins": int(r.red_wins),
                "draws": int(r.draws),
                "score": score,
                "avg": (score / games) if games else 0.0,
            })
        return rows

    @staticmethod
    def _safe_write(path, data):
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(data, indent=2))
        tmp.rename(path)

In [ ]:
for repeat_idx in range(n_repeats):
    if n_repeats > 1:
        repeat_dir = train_folder / f"repeat_{repeat_idx}"
        repeat_dir.mkdir(parents=False, exist_ok=False)
        print(
            f"\n{'=' * 80}\n=== Repeat {repeat_idx + 1}/{n_repeats} === (output: {repeat_dir})\n{'=' * 80}\n"
        )
    else:
        repeat_dir = train_folder

    # ── Initialise model ───────────────────────────────────────────────────
    torch._dynamo.reset()
    ntuple_net = (
        NTupleNetwork.load(pre_trained_model, device=device)
        if pre_trained_model
        else NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST_150).to(device)
    )
    if use_target_net:
        target_net = copy.deepcopy(ntuple_net)
        target_net.eval()
        for p in target_net.parameters():
            p.requires_grad_(False)
    else:
        target_net = ntuple_net

    if use_torch_compile:
        ntuple_net = torch.compile(ntuple_net)
        target_net = torch.compile(target_net)

    board = BoardBatch.empty(B, device)
    opt = torch.optim.Adam(ntuple_net.parameters(), lr=lr_initial)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lr_lambda=lambda step: (
            (lr_final / lr_initial) + (1 - lr_final / lr_initial) * gamma**step
        ),
    )
    logger = _TrainingLogger(
        repeat_dir, n_evaluate, n_truncate, n_repeats, repeat_idx, bitbully_agents
    )

    # ── Ring buffer (n_truncate + 1 slots) ────────────────────────────────
    buf_size = n_truncate + 1
    buf_all = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_active = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_moves = torch.zeros((buf_size, B), dtype=torch.int64, device=device)
    buf_V = torch.zeros((buf_size, B), dtype=torch.float32, device=device)
    buf_done = torch.zeros((buf_size, B), dtype=torch.bool, device=device)
    buf_reward = torch.zeros((buf_size, B), dtype=torch.float32, device=device)
    buf_random = torch.zeros((buf_size, B), dtype=torch.bool, device=device)

    loss = torch.tensor(float("nan"))
    V_pred = torch.zeros(B, device=device)
    update_mask = torch.zeros(B, dtype=torch.bool, device=device)

    for step in range(n_steps + 1):
        idx = step % buf_size

        # ── 1. Store current state ─────────────────────────────────────────
        buf_all[idx] = board.all_tokens
        buf_active[idx] = board.active_tokens
        buf_moves[idx] = board.moves_left
        with torch.no_grad():
            buf_V[idx] = target_net(board).float()

        # ── 2. Select action — ε-greedy afterstate search ─────────────────
        randomize = torch.rand((B,), device=device) < epsilon
        buf_random[idx] = randomize
        with torch.no_grad():
            best_mv, _ = best_afterstate_values(
                board,
                ntuple_net if use_online_net_for_action else target_net,
                randomize=randomize,
                use_non_losing=False,
            )

        # ── 3. Play move, record done & reward ────────────────────────────
        board.play_masks(best_mv)
        done = board.done()
        buf_done[idx] = done
        buf_reward[idx] = torch.where(
            done, board.reward().float(), torch.zeros(B, device=device)
        )

        # ── 4. Compute truncated λ-return and gradient step ───────────────
        if step >= n_truncate:
            oldest = (step - n_truncate) % buf_size

            with torch.no_grad():
                # Bootstrap from current value or terminal reward
                G = torch.where(done, board.reward().float(), target_net(board).float())
                # Fold in λ-weighted returns from oldest → newest
                for k in range(n_truncate - 1, -1, -1):
                    bk, bk1 = (oldest + k) % buf_size, (oldest + k + 1) % buf_size
                    G = torch.where(
                        buf_done[bk], buf_reward[bk], (1 - lam) * buf_V[bk1] + lam * G
                    )

            oldest_board = BoardBatch(
                buf_all[oldest], buf_active[oldest], buf_moves[oldest]
            )
            V_pred = ntuple_net(oldest_board)
            update_mask = buf_done[oldest] | ~buf_random[oldest]

            loss = F.mse_loss(V_pred[update_mask], G[update_mask])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            logger.snapshot_weights(ntuple_net, step)  # capture ΔW before update
            opt.step()
            scheduler.step()

        # ── 5. Polyak-update target network ───────────────────────────────
        if use_target_net:
            with torch.no_grad():
                for p_t, p_o in zip(target_net.parameters(), ntuple_net.parameters()):
                    p_t.data.mul_(1 - tau).add_(p_o.data, alpha=tau)

        # ── 6. Log metrics & run arena evaluation ─────────────────────────
        if step >= n_truncate:
            logger(
                step, ntuple_net, opt, loss, done, randomize, update_mask, V_pred, board
            )

        # ── 7. Reset finished games ────────────────────────────────────────
        board.reset(done)

    if n_repeats > 1:
        print(f"\n=== Repeat {repeat_idx + 1}/{n_repeats} finished ===\n")

print(f"\nAll {n_repeats} repeat(s) completed.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_adam_effective_lr(optimizer, title_suffix=""):
    """Plot histogram of Adam's per-parameter effective learning rates.

    For Adam, the actual update is:  W -= lr * m_hat / (sqrt(v_hat) + eps)
    So the effective learning rate per weight is:  lr / (sqrt(v_hat) + eps)
    where v_hat = v / (1 - beta2^t)  is the bias-corrected second moment.
    """
    for group_idx, group in enumerate(optimizer.param_groups):
        lr = group["lr"]
        beta1, beta2 = group["betas"]
        eps = group["eps"]

        for p_idx, p in enumerate(group["params"]):
            state = optimizer.state[p]
            if not state:
                print(f"No optimizer state yet (run at least 1 step first)")
                return

            step = state["step"]
            # exp_avg_sq is the raw (non-bias-corrected) second moment
            v = state["exp_avg_sq"]

            # Bias correction
            bias_correction2 = 1.0 - beta2**step

            # Effective LR per weight: lr / (sqrt(v / bias_correction2) + eps)
            v_hat = v / bias_correction2
            eff_lr = lr / (torch.sqrt(v_hat) + eps)

            # Only look at weights that have been touched (v > 0)
            mask = v > 0
            eff_lr_active = eff_lr[mask].detach().cpu().numpy()

            if eff_lr_active.size == 0:
                print("No active weights found.")
                return

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))

            # 1) Histogram of effective LR
            ax = axes[0]
            ax.hist(eff_lr_active, bins=100, edgecolor="black", alpha=0.7)
            ax.set_xlabel("Effective LR")
            ax.set_ylabel("Count")
            ax.set_title(f"Effective LR distribution (step {step})")
            ax.axvline(lr, color="r", linestyle="--", label=f"scheduled lr={lr:.2e}")
            ax.legend()

            # 2) Log-scale histogram
            ax = axes[1]
            log_eff = np.log10(eff_lr_active)
            ax.hist(log_eff, bins=100, edgecolor="black", alpha=0.7, color="orange")
            ax.set_xlabel("log10(Effective LR)")
            ax.set_ylabel("Count")
            ax.set_title(f"log10(Effective LR) distribution")
            ax.axvline(
                np.log10(lr),
                color="r",
                linestyle="--",
                label=f"log10(scheduled lr)={np.log10(lr):.2f}",
            )
            ax.legend()

            # 3) Ratio: effective_lr / scheduled_lr
            ax = axes[2]
            ratio = eff_lr_active / lr
            ax.hist(ratio, bins=100, edgecolor="black", alpha=0.7, color="green")
            ax.set_xlabel("Effective LR / Scheduled LR")
            ax.set_ylabel("Count")
            ax.set_title(f"Ratio effective/scheduled LR")
            ax.axvline(1.0, color="r", linestyle="--", label="ratio = 1")
            ax.legend()

            fig.suptitle(
                f"Adam effective LR analysis{title_suffix}\n"
                f"step={step}, scheduled_lr={lr:.2e}, "
                f"active weights={mask.sum().item()}/{v.numel()}, "
                f"eff_lr: median={np.median(eff_lr_active):.2e}, "
                f"min={eff_lr_active.min():.2e}, max={eff_lr_active.max():.2e}",
                fontsize=10,
            )
            plt.tight_layout()
            plt.show()

            # Also print summary stats of v_hat (the adapted squared gradient)
            v_hat_active = v_hat[mask].detach().cpu().numpy()
            print(
                f"v_hat (bias-corrected 2nd moment) stats for {mask.sum().item()} active weights:"
            )
            print(
                f"  min={v_hat_active.min():.2e}, max={v_hat_active.max():.2e}, "
                f"median={np.median(v_hat_active):.2e}, mean={v_hat_active.mean():.2e}"
            )


plot_adam_effective_lr(opt)